## Baseline

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

from narwhals import Categorical
from narwhals.selectors import categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_score, recall_score, ConfusionMatrixDisplay, confusion_matrix, roc_curve
from lightgbm.callback import early_stopping, log_evaluation


import lightgbm as lgb

In [4]:
path_to_data = "/home/usl/PycharmProjects/home-credit-default-risk/data/raw/home-credit-default-risk/"
# path_to_data = "/content/drive/MyDrive/Home_Credit_data/"

application_train_df = pd.read_csv(path_to_data+"application_train.csv")
application_test_df = pd.read_csv(path_to_data+"application_test.csv")
bureau_df = pd.read_csv(path_to_data+"bureau.csv")
bureau_balance_df = pd.read_csv(path_to_data+"bureau_balance.csv")
credit_card_balance_df = pd.read_csv(path_to_data+"credit_card_balance.csv")
home_credit_columns_description_df = pd.read_csv(path_to_data+"HomeCredit_columns_description.csv", encoding="cp1252")
installments_payments_df = pd.read_csv(path_to_data+"installments_payments.csv")
POS_CASH_balance_df = pd.read_csv(path_to_data+"POS_CASH_balance.csv")
previous_application_df = pd.read_csv(path_to_data+"previous_application.csv")

### Feature engineering

#### Main function for numerical and categorical features

### Numerical features

In [10]:
agg_func = ["median", "max"]
def make_numerical_features(df, id_col, numerical_cols, prefix):
    tables = []

    for feat in numerical_cols:
        temp = (df.groupby(id_col)[feat].agg(agg_func))
        temp.columns = [prefix + "_" + feat + "_" + func for func in agg_func]
        tables.append(temp)
    result = pd.concat(tables, axis=1)
    return result

In [12]:
numerical_cols_bureau_df = ["AMT_CREDIT_MAX_OVERDUE", "AMT_CREDIT_SUM", "AMT_CREDIT_SUM_DEBT","AMT_CREDIT_SUM_LIMIT", "AMT_CREDIT_SUM_OVERDUE", "AMT_ANNUITY"]

In [14]:
bureau_num = make_numerical_features(
    bureau_df,
    "SK_ID_CURR",
    numerical_cols_bureau_df,
    "BUREAU")
bureau_num

,BUREAU_AMT_CREDIT_MAX_OVERDUE_median,BUREAU_AMT_CREDIT_MAX_OVERDUE_max,BUREAU_AMT_CREDIT_SUM_median,BUREAU_AMT_CREDIT_SUM_max,BUREAU_AMT_CREDIT_SUM_DEBT_median,BUREAU_AMT_CREDIT_SUM_DEBT_max,BUREAU_AMT_CREDIT_SUM_LIMIT_median,BUREAU_AMT_CREDIT_SUM_LIMIT_max,BUREAU_AMT_CREDIT_SUM_OVERDUE_median,BUREAU_AMT_CREDIT_SUM_OVERDUE_max,BUREAU_AMT_ANNUITY_median,BUREAU_AMT_ANNUITY_max
SK_ID_CURR,,,,,,,,,,,,
100001,NaN,NaN,168345.00,378000.00,0.000,373239.00,0.0,0.000,0.0,0.0,0.0,10822.5
100002,40.500,5043.645,54130.50,450000.00,0.000,245781.00,0.0,31988.565,0.0,0.0,0.0,0.0
100003,0.000,0.000,92576.25,810000.00,0.000,0.00,0.0,810000.000,0.0,0.0,NaN,NaN
100004,0.000,0.000,94518.90,94537.80,0.000,0.00,0.0,0.000,0.0,0.0,NaN,NaN
100005,0.000,0.000,58500.00,568800.00,25321.500,543087.00,0.0,0.000,0.0,0.0,0.0,4261.5
...,...,...,...,...,...,...,...,...,...,...,...,...
456249,0.000,18945.000,248692.50,765000.00,0.000,163071.00,0.0,0.000,0.0,0.0,NaN,NaN
456250,0.000,0.000,483349.50,2153110.05,391731.615,1840308.48,0.0,58268.385,0.0,0.0,51799.5,384147.0
456253,NaN,NaN,675000.00,2250000.00,85518.000,1624797.00,0.0,0.000,0.0,0.0,58369.5,58369.5


### Categorical features

In [ ]:
def make_categorical_features(df, id_col, categorical_cols, prefix):
    result = (df[[id_col]].drop_duplicates().copy())

    for feat in categorical_cols:
        temp =

#### Bureau_df

In [5]:
bureau_df.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1716428 entries, 0 to 1716427
Data columns (total 17 columns):
 #   Column                  Non-Null Count    Dtype  
---  ------                  --------------    -----  
 0   SK_ID_CURR              1716428 non-null  int64  
 1   SK_ID_BUREAU            1716428 non-null  int64  
 2   CREDIT_ACTIVE           1716428 non-null  object 
 3   CREDIT_CURRENCY         1716428 non-null  object 
 4   DAYS_CREDIT             1716428 non-null  int64  
 5   CREDIT_DAY_OVERDUE      1716428 non-null  int64  
 6   DAYS_CREDIT_ENDDATE     1610875 non-null  float64
 7   DAYS_ENDDATE_FACT       1082775 non-null  float64
 8   AMT_CREDIT_MAX_OVERDUE  591940 non-null   float64
 9   CNT_CREDIT_PROLONG      1716428 non-null  int64  
 10  AMT_CREDIT_SUM          1716415 non-null  float64
 11  AMT_CREDIT_SUM_DEBT     1458759 non-null  float64
 12  AMT_CREDIT_SUM_LIMIT    1124648 non-null  float64
 13  AMT_CREDIT_SUM_OVERDUE  1716428 non-null  float64
 14  CR

In [6]:
bureau_description = home_credit_columns_description_df[home_credit_columns_description_df["Table"].astype(str).str.contains("bureau.csv", case = False, na=False)]
display(bureau_description[["Row", "Description"]])

,Row,Description
122,SK_ID_CURR,ID of loan in our sample - one loan in our sam...
123,SK_BUREAU_ID,Recoded ID of previous Credit Bureau credit re...
124,CREDIT_ACTIVE,Status of the Credit Bureau (CB) reported credits
125,CREDIT_CURRENCY,Recoded currency of the Credit Bureau credit
126,DAYS_CREDIT,How many days before current application did c...
127,CREDIT_DAY_OVERDUE,Number of days past due on CB credit at the ti...
128,DAYS_CREDIT_ENDDATE,Remaining duration of CB credit (in days) at t...
129,DAYS_ENDDATE_FACT,Days since CB credit ended at the time of appl...
130,AMT_CREDIT_MAX_OVERDUE,Maximal amount overdue on the Credit Bureau cr...
131,CNT_CREDIT_PROLONG,How many times was the Credit Bureau credit pr...


This is the first part of feature engineering. We will try to get new feature.

In [ ]:
bureau_credit_count = (bureau_df.groupby("SK_ID_CURR")["SK_ID_BUREAU"].count().rename("BUREAU_CREDIT_COUNT"))
bureau_credit_count.shape

In [ ]:
bureau_df["CREDIT_ACTIVE"].value_counts(dropna=False)

In [ ]:
bureau_df["IS_ACTIVE_CREDIT"] = (bureau_df["CREDIT_ACTIVE"] == "Active").astype(int)
bureau_active = (bureau_df.groupby("SK_ID_CURR").agg(BUREAU_ACTIVE_COUNT = ("IS_ACTIVE_CREDIT", "sum"), BUREAU_ACTIVE_SHARE = ("IS_ACTIVE_CREDIT", "mean")))
bureau_active.head()

In [ ]:
bureau_df["DAYS_CREDIT"].describe()


max indicates how recently the last loan was taken out,
min indicates how far back the credit history goes,
mean indicates average age of loans

In [ ]:
bureau_days_credit = (bureau_df.groupby("SK_ID_CURR").agg(BUREAU_LAST_CREDIT_DAYS = ("DAYS_CREDIT", "max"), BUREAU_FIRST_DAYS = ("DAYS_CREDIT", "min"), BUREAU_MEAN_DAYS = ("DAYS_CREDIT", "mean")))
bureau_days_credit.head()


We can obtain the length of the client's credit history:

In [ ]:
bureau_days_credit["BUREAU_CREDIT_HISTORY_LENGTH"] = (
    bureau_days_credit["BUREAU_LAST_CREDIT_DAYS"]
    - bureau_days_credit["BUREAU_FIRST_DAYS"])

In [ ]:
bureau_days_credit.head()

In [ ]:
bureau_days_credit.columns

In [ ]:
bureau_df["CREDIT_DAY_OVERDUE"].describe()

In [ ]:
bureau_df["IS_DAY_OVERDUE"] = (bureau_df["CREDIT_DAY_OVERDUE"] > 0).astype(int)

In [ ]:
bureau_overdue = (bureau_df.groupby("SK_ID_CURR").agg(BUREAU_MAX_DAYS_OVERDUE=("CREDIT_DAY_OVERDUE", "max"), BUREAU_OVERDUE_CREDIT_COUNT=("IS_DAY_OVERDUE", "sum"), BUREAU_OVERDUE_CREDIT_SHARE=("IS_DAY_OVERDUE", "mean")))

bureau_overdue.head(20)

In [ ]:
print("Доля записей с просрочкой:", (bureau_df["CREDIT_DAY_OVERDUE"] > 0).mean())
print("Количество записей с просрочкой:", (bureau_df["CREDIT_DAY_OVERDUE"] > 0).sum())

In [ ]:
(bureau_overdue["BUREAU_OVERDUE_CREDIT_COUNT"] > 0).mean()

In [ ]:
temp = application_train_df[["SK_ID_CURR", "TARGET"]].merge(bureau_overdue, on="SK_ID_CURR", how="left")
temp["HAS_BUREAU_OVERDUE"] = (temp["BUREAU_OVERDUE_CREDIT_COUNT"].fillna(0) > 0).astype(int)
temp.groupby("HAS_BUREAU_OVERDUE")["TARGET"].agg(["count", "mean"])

In [7]:
categorical_cols_bureau_df = ["CREDIT_ACTIVE", "CREDIT_CURRENCY", "CREDIT_TYPE"]
numerical_cols_bureau_df = ["AMT_CREDIT_MAX_OVERDUE", "AMT_CREDIT_SUM", "AMT_CREDIT_SUM_DEBT","AMT_CREDIT_SUM_LIMIT", "AMT_CREDIT_SUM_OVERDUE", "AMT_ANNUITY"]
special_cols_bureau_df = ["SK_ID_CURR", "SK_ID_BUREAU", "DAYS_CREDIT", "DAYS_CREDIT_ENDDATE","DAYS_ENDDATE_FACT", "DAYS_CREDIT_UPDATE", "CREDIT_DAY_OVERDUE", "CNT_CREDIT_PROLONG"]

In [8]:
print("SPECIAL:", special_cols_bureau_df)
print("\nCATEGORICAL:", categorical_cols_bureau_df)
print("\nNUMERICAL:", numerical_cols_bureau_df)

SPECIAL: ['SK_ID_CURR', 'SK_ID_BUREAU', 'DAYS_CREDIT', 'DAYS_CREDIT_ENDDATE', 'DAYS_ENDDATE_FACT', 'DAYS_CREDIT_UPDATE', 'CREDIT_DAY_OVERDUE', 'CNT_CREDIT_PROLONG']

CATEGORICAL: ['CREDIT_ACTIVE', 'CREDIT_CURRENCY', 'CREDIT_TYPE']

NUMERICAL: ['AMT_CREDIT_MAX_OVERDUE', 'AMT_CREDIT_SUM', 'AMT_CREDIT_SUM_DEBT', 'AMT_CREDIT_SUM_LIMIT', 'AMT_CREDIT_SUM_OVERDUE', 'AMT_ANNUITY']


In [9]:
agg_func = ["median", "max"]

In [ ]:
def make_numerical_feat():
    tadles = []
    for feat in numerical_cols:



In [ ]:
# transf_tables_bureau = []
#
# for feat in numerical_cols_bureau_df:
#     transf_bureau = bureau_df.groupby("SK_ID_CURR")[feat].agg(agg_func)
#     transf_bureau.columns = [feat+"_"+func for func in agg_func]
#     transf_tables_bureau.append(transf_bureau)
# bureau_transformed = pd.concat(transf_tables_bureau, axis=1)
# bureau_transformed


In [ ]:
# application_train_df = application_train_df.merge(bureau_transformed, on="SK_ID_CURR", how="left")
# application_test_df = application_test_df.merge(bureau_transformed, on="SK_ID_CURR", how="left")
# application_train_df

In [ ]:
transf_tables_bureau = []

for feat in categorical_cols_bureau_df:
    transf_bureau = bureau_df.groupby(["SK_ID_CURR", feat])[["SK_ID_BUREAU"]].count()
    transf_bureau.columns = [feat+"_count"]
    transf_bureau = transf_bureau.reset_index()
    transf_bureau = transf_bureau.set_index("SK_ID_CURR")
    transf_tables_bureau.append(transf_bureau)

    display(transf_bureau)
#bureau_transformed = pd.concat(transf_tables_bureau, axis=1)
#bureau_transformed


In [ ]:
transf_tables_bureau = []

for feat in categorical_cols_bureau_df:
    transf_bureau = bureau_df.pivot_table(index="SK_ID_CURR", columns=feat, aggfunc="count", fill_value=0, values="SK_ID_BUREAU")
    #transf_tables_bureau.append(transf_bureau)

    display(transf_bureau)
#bureau_transformed = pd.concat(transf_tables_bureau, axis=1)
#bureau_transformed

## On the class

In [ ]:
bureau_df.groupby(["SK_ID_CURR", "CREDIT_ACTIVE"])[["SK_ID_BUREAU"]].count()

In [ ]:
bureau_days_credit_mean = bureau_df.groupby("SK_ID_CURR")[["DAYS_CREDIT"]].mean()
bureau_days_credit_mean

In [ ]:
bureau_days_credit_median = bureau_df.groupby("SK_ID_CURR")[["DAYS_CREDIT"]].median()
bureau_days_credit_median

In [ ]:
bureau_days_credit_mean.merge(bureau_days_credit_median, on="SK_ID_CURR")

#### previous_application_df

In [ ]:
previous_application_df = pd.read_csv(path_to_data+"previous_application.csv")

In [ ]:
previous_application_df.info(show_counts=True)
previous_application_df.shape

In [ ]:
home_credit_columns_description_df = pd.read_csv(path_to_data+"HomeCredit_columns_description.csv", encoding="cp1252")
previous_application_description = home_credit_columns_description_df[home_credit_columns_description_df["Table"].astype(str).str.contains("previous_application.csv", case = False, na=False)]
display(previous_application_description[["Row", "Description"]])

In [ ]:
categorical_cols_prev_app_df = ["NAME_CONTRACT_TYPE", "WEEKDAY_APPR_PROCESS_START","FLAG_LAST_APPL_PER_CONTRACT", "NAME_CASH_LOAN_PURPOSE", "NAME_CONTRACT_STATUS","NAME_PAYMENT_TYPE", "CODE_REJECT_REASON", "NAME_TYPE_SUITE", "NAME_CLIENT_TYPE","NAME_GOODS_CATEGORY", "NAME_PORTFOLIO", "NAME_PRODUCT_TYPE", "CHANNEL_TYPE","NAME_SELLER_INDUSTRY", "NAME_YIELD_GROUP", "PRODUCT_COMBINATION"]
numerical_cols_prev_app_df = ["AMT_ANNUITY", "AMT_APPLICATION", "AMT_CREDIT","AMT_DOWN_PAYMENT", "AMT_GOODS_PRICE", "RATE_DOWN_PAYMENT", "RATE_INTEREST_PRIMARY", "RATE_INTEREST_PRIVILEGED"]
special_cols_prev_app_df = ["SK_ID_PREV", "SK_ID_CURR", "HOUR_APPR_PROCESS_START", "NFLAG_LAST_APPL_IN_DAY", "CNT_PAYMENT", "DAYS_DECISION", "DAYS_FIRST_DRAWING", "DAYS_FIRST_DUE", "DAYS_LAST_DUE_1ST_VERSION", "DAYS_LAST_DUE", "DAYS_TERMINATION", "NFLAG_INSURED_ON_APPROVAL", "SELLERPLACE_AREA"]

In [ ]:
print("SPECIAL:", special_cols_prev_app_df)
print("\nCATEGORICAL:", categorical_cols_prev_app_df)
print("\nNUMERICAL:", numerical_cols_prev_app_df)

#### bureau_balance_df

In [ ]:
bureau_balance_df = pd.read_csv(path_to_data+"bureau_balance.csv")

In [ ]:
bureau_balance_df.info(show_counts=True)
bureau_balance_df.shape

In [ ]:
home_credit_columns_description_df = pd.read_csv(path_to_data+"HomeCredit_columns_description.csv", encoding="cp1252")
bureau_balance_description = home_credit_columns_description_df[home_credit_columns_description_df["Table"].astype(str).str.contains("bureau_balance.csv", case = False, na=False)]
display(bureau_balance_description[["Row", "Description"]])

In [ ]:
categorical_cols_bureau_balance_df = ["STATUS"]
numerical_cols_bureau_balance_df = []
special_cols_bureau_balance_df = ["SK_ID_BUREAU", "MONTHS_BALANCE"]

In [ ]:
print("SPECIAL:", special_cols_bureau_balance_df)
print("\nCATEGORICAL:", categorical_cols_bureau_balance_df)
print("\nNUMERICAL:", numerical_cols_bureau_balance_df)

#### POS_CASH_balance_df

In [ ]:
POS_CASH_balance_df = pd.read_csv(path_to_data+"POS_CASH_balance.csv")

In [ ]:
POS_CASH_balance_df.info(show_counts=True)
POS_CASH_balance_df.shape

In [ ]:
home_credit_columns_description_df = pd.read_csv(path_to_data+"HomeCredit_columns_description.csv", encoding="cp1252")
POS_CASH_balance_description = home_credit_columns_description_df[home_credit_columns_description_df["Table"].astype(str).str.contains("POS_CASH_balance.csv", case = False, na=False)]
display(POS_CASH_balance_description[["Row", "Description"]])


In [ ]:
categorical_cols_POS_CASH_bal_df = ["NAME_CONTRACT_STATUS"]
numerical_cols_POS_CASH_bal_df = ["CNT_INSTALMENT", "CNT_INSTALMENT_FUTURE"]
special_cols_POS_CASH_bal_df = ["SK_ID_PREV", "SK_ID_CURR", "MONTHS_BALANCE", "SK_DPD", "SK_DPD_DEF"]

In [ ]:
print("SPECIAL:", special_cols_POS_CASH_bal_df)
print("\nCATEGORICAL:", categorical_cols_POS_CASH_bal_df)
print("\nNUMERICAL:", numerical_cols_POS_CASH_bal_df)

#### installments_payments_df

In [ ]:
installments_payments_df = pd.read_csv(path_to_data+"installments_payments.csv")

In [ ]:
installments_payments_df.info(show_counts=True)
installments_payments_df.shape

In [ ]:
home_credit_columns_description_df = pd.read_csv(path_to_data+"HomeCredit_columns_description.csv", encoding="cp1252")
inst_payment_description = home_credit_columns_description_df[home_credit_columns_description_df["Table"].astype(str).str.contains("installments_payments.csv", case = False, na=False)]
display(inst_payment_description[["Row", "Description"]])

In [ ]:
categorical_cols_inst_payment_df = []
numerical_cols_inst_payment_df = ["AMT_INSTALMENT", "AMT_PAYMENT"]
special_cols_inst_payment_df = ["SK_ID_PREV", "SK_ID_CURR", "NUM_INSTALMENT_VERSION", "NUM_INSTALMENT_NUMBER", "DAYS_INSTALMENT", "DAYS_ENTRY_PAYMENT"]

In [ ]:
print("SPECIAL:", special_cols_inst_payment_df)
print("\nCATEGORICAL:", categorical_cols_inst_payment_df)
print("\nNUMERICAL:", numerical_cols_inst_payment_df)

#### credit_card_balance_df

In [ ]:
credit_card_balance_df = pd.read_csv(path_to_data+"credit_card_balance.csv")

In [ ]:
credit_card_balance_df.info(show_counts=True)
credit_card_balance_df.shape

In [ ]:
home_credit_columns_description_df = pd.read_csv(path_to_data+"HomeCredit_columns_description.csv", encoding="cp1252")
credit_card_balance_description = home_credit_columns_description_df[home_credit_columns_description_df["Table"].astype(str).str.contains("credit_card_balance.csv", case = False, na=False)]
display(credit_card_balance_description[["Row", "Description"]])

In [ ]:
categorical_cols_credit_card_balance_df = ["NAME_CONTRACT_STATUS"]
numerical_cols_credit_card_balance_df = ["AMT_BALANCE", "AMT_CREDIT_LIMIT_ACTUAL", "AMT_DRAWINGS_ATM_CURRENT", "AMT_DRAWINGS_CURRENT", "AMT_DRAWINGS_OTHER_CURRENT", "AMT_DRAWINGS_POS_CURRENT", "AMT_INST_MIN_REGULARITY", "AMT_PAYMENT_CURRENT", "AMT_PAYMENT_TOTAL_CURRENT", "AMT_RECEIVABLE_PRINCIPAL", "AMT_RECIVABLE", "AMT_TOTAL_RECEIVABLE", "CNT_DRAWINGS_ATM_CURRENT", "CNT_DRAWINGS_CURRENT", "CNT_DRAWINGS_OTHER_CURRENT", "CNT_DRAWINGS_POS_CURRENT", "CNT_INSTALMENT_MATURE_CUM"]
special_cols_credit_card_balance_df = ["SK_ID_PREV", "SK_ID_CURR", "MONTHS_BALANCE", "SK_DPD", "SK_DPD_DEF"]

In [ ]:
print("SPECIAL:", special_cols_credit_card_balance_df)
print("\nCATEGORICAL:", categorical_cols_credit_card_balance_df)
print("\nNUMERICAL:", numerical_cols_credit_card_balance_df)

Split into train and val

In [ ]:
train_users = application_train_df["SK_ID_CURR"].unique()
train_ids, val_ids = train_test_split(train_users, test_size=0.2, random_state=42)
train_ids

In [ ]:
train_df = application_train_df[application_train_df.SK_ID_CURR.isin(train_ids)].copy()
val_df = application_train_df[application_train_df.SK_ID_CURR.isin(val_ids)].copy()

In [ ]:
train_df.shape, val_df.shape

In [ ]:
application_train_df["TARGET"].mean(), train_df["TARGET"].mean(), val_df["TARGET"].mean()

In [ ]:
drop_feat = ["SK_ID_CURR"]
train_df.drop(drop_feat, axis=1, inplace=True)
val_df.drop(drop_feat, axis=1, inplace=True)
application_test_df.drop(drop_feat, axis=1, inplace=True)

In [ ]:
X_train = train_df.drop("TARGET", axis=1)
y_train = train_df["TARGET"]
X_val = val_df.drop("TARGET", axis=1)
y_val = val_df["TARGET"]

In [ ]:
cat_cols = X_train.select_dtypes(include = 'object').columns.tolist()
len(cat_cols)

In [ ]:
for col in cat_cols:
    categories = X_train[col].dropna().unique()
    X_train[col] = pd.Categorical(X_train[col], categories = categories)
    X_val[col] = pd.Categorical(X_val[col], categories = categories)


In [ ]:
X_train.select_dtypes(include='object').columns

### LightGBM

In [ ]:
lgb_clf = lgb.LGBMClassifier(n_estimators=1000, max_depth=4, learning_rate=0.1, random_state = 42, n_jobs=-1)
lgb_clf.fit(X_train, y_train, eval_set = [(X_val, y_val)], eval_metric='auc', categorical_feature=cat_cols, callbacks = [early_stopping(stopping_rounds=100), log_evaluation(period = 100)])

threshold = 0.5
val_pred_proba = lgb_clf.predict_proba(X_val)[:, 1]
val_pred = (val_pred_proba >= threshold).astype(int)


roc_auc_lgb = roc_auc_score(y_val, val_pred_proba)
prec_lgb = precision_score(y_val, val_pred, zero_division=0)
rec_lgb = recall_score(y_val, val_pred, zero_division=0)

print("Best_iteration:", lgb_clf.best_iteration_)
print("ROC-AUC_lgb:", roc_auc_lgb)
print("Precision_lgb:", prec_lgb)
print("Recall_lgb:", rec_lgb)

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_val, val_pred)
plt.title("Confusion Matrix LightGBM")

And now we will calculate the roc_curve and plot a graph:

In [ ]:
fpr, tpr, thresholds = roc_curve(y_val, val_pred_proba)
plt.figure(figsize=(7, 6))

plt.plot(fpr, tpr, label = f"LightGBM ROC-AUC={roc_auc_lgb:.4f}")
plt.plot([0, 1], [0, 1], 'k--', label = "Random classifier ROC-AUC=0.5")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.0])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC curve LightGBM")
plt.legend(loc="lower right")
plt.grid()

We will prepare the test part for training:

In [ ]:
application_test_df.shape

In [ ]:
X_test = application_test_df

In [ ]:
for col in cat_cols:
    X_test[col] = pd.Categorical(X_test[col], categories = X_train[col].cat.categories)
X_test.shape, X_train.shape

In [ ]:
list(X_train.columns) == list(X_test.columns)

In [ ]:
test_pred_proba = lgb_clf.predict_proba(X_test)[:, 1]

In [ ]:
submission = sample_submission_df.copy()
submission["TARGET"] = test_pred_proba
submission.head()

In [ ]:
submission.shape, application_test_df.shape

In [ ]:
submission.to_csv("submission_lgb_baseline.csv", index=False)